# Análise de Furtos e Roubos na Grande Vitória

**Disciplina:** Análise de Dados — Projeto Integrador III  

**Grupo:**  Alexsander, Ester, Larissa Moraes, Lucas Rufino, Marcelo Mindas, Vanderson de Almeida.

**Data:** Maio de 2026 


---

## 1. Introdução

Este notebook apresenta uma análise exploratória dos dados de furtos e roubos
registrados na região da Grande Vitória (ES), obtidos no portal [SESP](https://sesp.es.gov.br/painel-de-crimes-contra-o-patrimonio)

O objetivo é identificar padrões temporais, geográficos e por tipo de ocorrência,
respondendo às seguintes perguntas:

1. Qual município concentra mais ocorrências?
2. Como os crimes se distribuem ao longo dos meses?
3. Em quais horários as ocorrências são mais frequentes?
4. Existe diferença no padrão entre furtos e roubos?

In [1]:
import pandas as pd

In [2]:
dados = pd.read_excel("../dados/brutos/MICRODADOS_OCORRENCIAS.xlsx")
dados.sample(1000)

,Data,Hora,TipoIncidente,TipoLocal,Municipio,Bairro
98174,2025-01-23,14:46:00,ESTELIONATO/FRAUDE,NaN,VILA VELHA,ATAIDE
367701,2021-09-22,07:00:00,ESTELIONATO/FRAUDE,NaN,SERRA,MORADA DE LARANJEIRAS
199262,2023-10-10,20:00:00,FURTO: EM RESIDÊNCIA/CONDOMÍNIO,RESIDÊNCIA,ITAPEMIRIM,OUTRO LOCAL
544657,2019-01-15,14:00:00,FURTO: A PESSOA EM VIA PÚBLICA,VIA PÚBLICA,SERRA,ESTANCIA MONAZITICA
562865,2018-10-09,Indeterminada,FURTO: EM TRANSPORTE COLETIVO,NaN,LINHARES,INTERLAGOS
...,...,...,...,...,...,...
611890,2018-01-30,08:00:00,FURTO: EM TRANSPORTE COLETIVO,NaN,CARIACICA,CAMPO GRANDE
567654,2018-09-14,11:00:00,ROUBO: A PESSOA EM VIA PÚBLICA,VIA PÚBLICA,VILA VELHA,COQUEIRAL DE ITAPARICA
374279,2021-08-20,10:00:00,ESTELIONATO/FRAUDE,COMÉRCIO,SERRA,PARQUE RESIDENCIAL LARANJEIRAS
69197,2025-06-05,07:50:00,FURTO: EM TRANSPORTE COLETIVO,TRANSP COLETIVO,VITORIA,SANTA LUIZA


In [3]:
dados.columns

Index(['Data', 'Hora', 'TipoIncidente', 'TipoLocal', 'Municipio', 'Bairro'], dtype='str')

In [4]:
dados.shape

(617940, 6)

In [5]:
dados.info

<bound method DataFrame.info of              Data           Hora                         TipoIncidente  \
0      2026-04-30       10:00:00                    ESTELIONATO/FRAUDE   
1      2026-04-30       01:20:00        ROUBO: A PESSOA EM VIA PÚBLICA   
2      2026-04-30       13:41:00                    ESTELIONATO/FRAUDE   
3      2026-04-30  Indeterminada                    ESTELIONATO/FRAUDE   
4      2026-04-30  Indeterminada                    ESTELIONATO/FRAUDE   
...           ...            ...                                   ...   
617935 2018-01-01       17:13:00       FURTO: EM RESIDÊNCIA/CONDOMÍNIO   
617936 2018-01-01       20:04:00   FURTO: EM ESTABELECIMENTO COMERCIAL   
617937 2018-01-01       06:30:00   FURTO: EM ESTABELECIMENTO COMERCIAL   
617938 2018-01-01  Indeterminada        ROUBO: A PESSOA EM VIA PÚBLICA   
617939 2018-01-01       16:11:00        ROUBO: A PESSOA EM VIA PÚBLICA   

               TipoLocal                Municipio                 Bairro  
0   

In [6]:
dados.isnull().sum()

Data                 0
Hora                 0
TipoIncidente        0
TipoLocal        82204
Municipio            0
Bairro               0
dtype: int64

## Filtragem inicial da base de dados

In [7]:
# Padronização de letras maiúsculas/minúsculas e espaços
dados["TipoIncidente"] = dados["TipoIncidente"].str.upper().str.strip()
dados["Municipio"] = dados["Municipio"].str.upper().str.strip()
dados["Bairro"] = dados["Bairro"].str.upper().str.strip()
dados["TipoLocal"] = dados["TipoLocal"].str.upper().str.strip()

In [8]:
# Conversão de datas
dados["Data"] = pd.to_datetime(dados["Data"], errors="coerce")

In [9]:
# Período de tempo 
dt_inicio = pd.Timestamp("2021-03-01")
dt_fim = pd.Timestamp("2026-03-30")

dados_filtr = dados[
    (dados["Data"] >= dt_inicio) &
    (dados["Data"] <= dt_fim)
].copy()

In [10]:
# Filtragem dos Municípios
municipios_gv = [
    "VITORIA",
    "VILA VELHA",
    "SERRA",
    "CARIACICA",
    "VIANA",
    "GUARAPARI",
    "FUNDAO"
]
dados_filtr = dados_filtr[
    dados_filtr["TipoIncidente"].isin(municipios_gv)
].copy()

In [11]:
# Filtragem de Furtos e Roubos
dados_filtr = dados_filtr[
    dados_filtr ["TipoIncidente"].str.contains("FURTO|ROUBO", na=False)
].copy()

In [12]:
# Categoria resumida
dados_filtr["CategoriaCrime"] = dados_filtr["TipoIncidente"].apply(
    lambda tipo: "Furto" if "FURTO" in tipo else "Roubo"
)

In [13]:
# Variáveis de tempo
dados_filtr["Ano"] = dados_filtr["Data"].dt.year
dados_filtr["Mes"] = dados_filtr["Data"].dt.month
dados_filtr["AnoMes"] = dados_filtr["Data"].dt.to_period("M")
dados_filtr["DiaSemana"] = dados_filtr["Data"].dt.day_name()

traducao_dias = {
    "Monday": "Segunda-feira",
    "Tuesday": "Terça-feira",
    "Wednesday": "Quarta-feira",
    "Thursday": "Quinta-feira",
    "Friday": "Sexta-feira",
    "Saturday": "Sábado",
    "Sunday": "Domingo"
}

dados_filtr["DiaSemana"] = dados_filtr["DiaSemana"].map(traducao_dias)


In [14]:
# Tratamento de Hora
dados_filtr["HoraConvertida"] = pd.to_datetime(
    dados_filtr["Hora"],
    format="%H:%M:%S",
    errors="coerce"
)

dados_filtr["HoraDia"] = dados_filtr["HoraConvertida"].dt.hour

In [15]:
dados_hora = dados_filtr.dropna(subset=["HoraDia"]).copy()